In [0]:
from datetime import datetime
import os.path

today=datetime.now().strftime("%Y%m%d")

spotify_path = "/Volumes/workspace/default/spotify"
full_path=f"{spotify_path}/spotify_songs_{today}.parquet"

print(f"🔍 Checking for uploaded files for date: {today} ")
print("\n📁 Spotify files:")
try:
    exists = os.path.isfile(full_path)
    if exists:
        print("✅ Found Spotify data")
    else:
        print("❌ No Spotify data found")
except Exception as e:
    print(f"Error when trying to find Spotify data: {e}")
    exit(-1)


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

bronze_spotify_table = "main.default.bronze_spotify_songs"

def load_spotify_to_bronze():
    try:
        spotify_raw = spark.read.parquet(spotify_path)
        print(f"📊 Raw YouTube files loaded: {spotify_raw.count():,} records")

        spotify_bronze = spotify_raw \
            .withColumn("bronze_load_timestamp", current_timestamp()) \
            .withColumn("data_source", lit("spotify_api")) \
            .withColumn("file_path", input_file_name())

        spotify_bronze.write.mode("overwrite").saveAsTable(bronze_spotify_table)
        
        final_count = spark.table(bronze_spotify_table).count()
        print(f"✅ Spotify Bronze table created: {final_count:,} records")
        
        # Show sample
        print("\n📺 Sample Spotify Bronze records:")
        spark.table(bronze_spotify_table).select(
            "name", "artist_name", "album_name", "popularity", "bronze_load_timestamp"
        ).show(3, truncate=False)
        
        return True
        
    except Exception as e:
        print(f"Error loading Spotify files: {e}")

In [0]:
spotify_data_loaded = load_spotify_to_bronze()

In [0]:
%run ../utils/bronze_ingestion_data_quality_report

if spotify_data_loaded:
    bronze_data_quality_report(bronze_youtube_table, "video_id", ["video_id", "title", "channel_title"])